In [ ]:
import os
from ray import tune
from ray.tune.search.optuna import OptunaSearch
from datetime import datetime
import shutil
from ultralytics import YOLO
import yaml

print("DONE")

In [22]:
# Percorso originale del dataset
dataset_root = "/kaggle/input/datasets/antonioquattromini/audiolabv2/AudioLabs_v2"
original_yaml = os.path.join(dataset_root, "data.yaml")

# Leggi il file
with open(original_yaml, 'r') as f:
    config = yaml.safe_load(f)

# Imposta il percorso assoluto
config['path'] = dataset_root
config['train'] = 'images/train'
config['val'] = 'images/val'

# Salva in una posizione scrivibile
fixed_yaml = "/kaggle/working/data_fixed.yaml"
with open(fixed_yaml, 'w') as f:
    yaml.dump(config, f)

print(f"✅ data.yaml corretto salvato in {fixed_yaml}")
print("Contenuto:")
with open(fixed_yaml, 'r') as f:
    print(f.read())

# === CONFIGURAZIONE PER KAGGLE ===
DATA_PATH = fixed_yaml


# Crea directory per i risultati
os.makedirs("/kaggle/working/tuning_results", exist_ok=True)

# === MODELLO ===
model = YOLO("yolo11n.pt")

# === SPAZIO DI RICERCA (stesso) ===
search_space = {
    "optimizer": "adamw",
    "lr0": tune.uniform(0.0001, 0.01),
    "momentum": tune.uniform(0.85, 0.98),
    "weight_decay": tune.uniform(0.0, 0.001),
    "box": tune.uniform(4.0, 8.0),
    "cls": tune.uniform(0.8, 2.0),
    "dfl": tune.uniform(1.0, 2.0),
}

# === ALGORITMO DI RICERCA: Optuna (bayesiano, più efficiente del casuale) ===
search_alg = OptunaSearch(metric="metrics/mAP50-95(B)", mode="max")

# === NOME UNIVOCO PER EVITARE SOVRASCRITTURE ===
run_name = f"tuning_optuna_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

# === TUNING ===
model.tune(
    data=DATA_PATH,
    use_ray=True,
    space=search_space,
    search_alg=search_alg,      # Ottimizzazione bayesiana
    epochs=15,
    grace_period=5,
    gpu_per_trial=1,            # GPU
    iterations=15,
    resume=True,  
    name="tuning_optuna_20260528_214326",
    batch=16,
    workers=8
)

# === COPIA RISULTATI IN DIRECTORY PERSISTENTE ===
source_dir = f"/kaggle/working/{run_name}"
if os.path.exists(source_dir):
    shutil.copytree(source_dir, "/kaggle/working/tuning_results", dirs_exist_ok=True)
    print(f"Risultati salvati in /kaggle/working/tuning_results")
else:
    print("Attenzione: directory dei risultati non trovata.")

2026-05-28 21:58:02,239	INFO tune_controller.py:441 -- Restoring the run from the latest experiment state file: experiment_state-2026-05-28_21-43-35.json


(_tune pid=2107) Ultralytics 8.4.56 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
(_tune pid=2107) engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=5.787099718057663, cache=False, cfg=None, classes=None, close_mosaic=10, cls=1.1907873292777975, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data_fixed.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.7528681514123698, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=15, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.006809771937862167, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.8760003456673272, mosaic=1.0, multi_scale=0.0, n

(_tune pid=2107) libpng warning: iCCP: profile 'icc': 'RGB ': RGB color space not permitted on grayscale PNG


(_tune pid=2107) val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 215.1±76.5 MB/s, size: 281.6 KB)


(_tune pid=2107) libpng warning: iCCP: profile 'icc': 'RGB ': RGB color space not permitted on grayscale PNG


...

       5/15      5.27G     0.5429      1.595      1.157         15        640: 100% ━━━━━━━━━━━━ 46/46 4.3it/s 10.8s0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 17% ━━────────── 1/6 1.5s/it 0.4s<7.5s


(_tune pid=3717) libpng warning: iCCP: profile 'icc': 'RGB ': RGB color space not permitted on grayscale PNG


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 33% ━━━━──────── 2/6 1.3it/s 0.8s<3.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 50% ━━━━━━────── 3/6 1.7it/s 1.2s<1.8s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 67% ━━━━━━━━──── 4/6 1.9it/s 1.6s<1.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 83% ━━━━━━━━━━── 5/6 2.2it/s 2.0s<0.5s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 2.7it/s 2.2s
(_tune pid=3717)                    all        184      17850      0.725       0.88      0.825      0.488


2026-05-28 22:36:01,864	INFO tune.py:1001 -- Wrote the latest version of all result files and experiment state to '/kaggle/working/runs/detect/tuning_optuna_20260528_214326' in 0.0117s.
2026-05-28 22:36:01,873	INFO tune.py:1033 -- Total run time: 2279.65 seconds (2279.59 seconds for the tuning loop).
(_tune pid=3717) /usr/local/lib/python3.12/dist-packages/ray/train/_internal/session.py:791: RayDeprecationWarning: `ray.train.report` should be switched to `ray.tune.report` when running in a function passed to Ray Tune. This will be an error in the future. See this issue for more context: https://github.com/ray-project/ray/issues/49454
(_tune pid=3717)   _log_deprecation_warning(


Attenzione: directory dei risultati non trovata.
